# tree.ipynb

Notebook 内でディレクトリ構造を確認するための、軽量な `tree()` 関数です。

Linux の `tree` コマンド完全互換ではなく、Notebook 上で読みやすく使いやすい Python 関数 API として実装しています。

```python
tree(path=".", exclude=[".git", "node_modules"], max_depth=3)
```


## 実装方針

- `tree(path, exclude=..., max_depth=...)` の形で使う。
- CLI 風の `-I`、`-L`、`-a`、`-f` などは実装しない。
- 除外ディレクトリには入らない。
- Notebook で上から読んで理解しやすいように、処理の流れを `tree()` の中に素直に書く。
- validation は、存在確認とディレクトリ確認の最小限にする。


In [1]:
from pathlib import Path

def tree(
    path=".",
    exclude=None,
    max_depth=None,
    show_files=True,
    show_hidden=False,
    ascii_only=False,
):
    """Print a directory tree for use inside a notebook.

    Parameters
    ----------
    path : str or pathlib.Path
        Root directory to display.
    exclude : list[str] or set[str] or tuple[str], optional
        Directory or file names to skip. Matching is by name, not by full path.
    max_depth : int, optional
        Maximum depth below the root. ``max_depth=1`` shows only the root's direct children.
        ``None`` means no depth limit.
    show_files : bool
        If False, show directories only.
    show_hidden : bool
        If False, skip names starting with ``.``.
    ascii_only : bool
        If True, use ASCII branches instead of box-drawing characters.
    """
    root = Path(path)
    exclude = set(exclude or [])

    if not root.exists():
        raise FileNotFoundError(f"Path not found: {root}")

    if not root.is_dir():
        raise NotADirectoryError(f"Path is not a directory: {root}")

    branch_mid = "+-- " if ascii_only else "├── "
    branch_last = "`-- " if ascii_only else "└── "
    indent_mid = "|   " if ascii_only else "│   "
    indent_last = "    "

    print(root.resolve())

    def print_children(directory, prefix="", depth=0):
        if max_depth is not None and depth >= max_depth:
            return

        items = []

        for item in directory.iterdir():
            if item.name in exclude:
                continue

            if not show_hidden and item.name.startswith("."):
                continue

            if item.is_file() and not show_files:
                continue

            items.append(item)

        items.sort(key=lambda item: (item.is_file(), item.name.lower()))

        for index, item in enumerate(items):
            is_last = index == len(items) - 1
            branch = branch_last if is_last else branch_mid

            print(f"{prefix}{branch}{item.name}")

            if item.is_dir():
                next_prefix = prefix + (indent_last if is_last else indent_mid)
                print_children(item, next_prefix, depth + 1)

    print_children(root)


## 使用例

### カレントディレクトリを表示


In [2]:
tree(".")

C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP
├── 99_old
│   ├── Pod Plane
│   │   ├── a.csv
│   │   ├── Pod.xlsm
│   │   ├── podplane.vsp3
│   │   ├── podplane_CompGeom.csv
│   │   ├── podplane_CompGeom.txt
│   │   ├── podplane_DegenGeom.adb
│   │   ├── podplane_DegenGeom.adb.cases
│   │   ├── podplane_DegenGeom.case.-11.quad.1.dat
│   │   ├── podplane_DegenGeom.case.-15.quad.1.dat
│   │   ├── podplane_DegenGeom.case.-7.quad.1.dat
│   │   ├── podplane_DegenGeom.case.-9.quad.1.dat
│   │   ├── podplane_DegenGeom.case.1.quad.1.dat
│   │   ├── podplane_DegenGeom.case.10.quad.1.dat
│   │   ├── podplane_DegenGeom.case.11.quad.1.dat
│   │   ├── podplane_DegenGeom.case.12.quad.1.dat
│   │   ├── podplane_DegenGeom.case.13.quad.1.dat
│   │   ├── podplane_DegenGeom.case.14.quad.1.dat
│   │   ├── podplane_DegenGeom.case.2.quad.1.dat
│   │   ├── podplane_DegenGeom.case.3.quad.1.dat
│   │   ├── podplane_DegenGeom.case.4.quad.1.dat
│   │   ├── podplane_DegenGeom.case.5.quad.1.dat
│   │   ├── podpla

### よく除外するディレクトリを指定


In [8]:
tree(".", exclude=[".git", "node_modules", "__pycache__", ".ipynb_checkpoints", "99_old", "vv_gamma_cases"])

C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP
├── docs
├── examples
│   ├── models
│   │   ├── DG1000s
│   │   │   └── DG1000S-Flight-Manual.pdf
│   │   ├── DG300
│   │   │   └── DG300_Flignt-Manual.pdf
│   │   ├── G102
│   │   │   └── Grob-G102-Flight-Manual.pdf
│   │   └── G103A
│   │       ├── airfoil
│   │       │   ├── E603.csv
│   │       │   ├── E603.dat
│   │       │   └── E603_CST.txt
│   │       ├── view
│   │       │   ├── G103A.png
│   │       │   ├── G103A_FRONT.png
│   │       │   ├── G103A_FRONT_Y.png
│   │       │   ├── G103A_LEFT.png
│   │       │   ├── G103A_LEFT_Y.png
│   │       │   ├── G103A_TOP.png
│   │       │   ├── G103A_TOP_Y.png
│   │       │   └── G103A_TOP_Y_Zoom.png
│   │       ├── G103A.adb
│   │       ├── G103A.adb.cases
│   │       ├── G103A.ALL.taglist
│   │       ├── G103A.ControlSurfaces.taglist
│   │       ├── G103A.csf
│   │       ├── G103A.flt
│   │       ├── G103A.group.0
│   │       ├── G103A.group.1
│   │       ├── G103A.history
│   │       ├─

### 深さを制限

`max_depth=2` の場合、ルート直下から2階層分だけ表示します。


In [4]:
tree(".", exclude=[".git", "node_modules", "__pycache__"], max_depth=2)

C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP
├── 99_old
│   ├── Pod Plane
│   ├── pyvsp.2026.06.28
│   ├── tests.2026.06.28
│   ├── tmp.2026.06.28
│   ├── VSPU Tutorial Models
│   ├── AnalysisVSPAERO.2026.07.03.py
│   ├── TurnTrim.2026.07.03.py
│   └── VVGammaChart.2026.07.03.py
├── docs
├── examples
│   ├── models
│   ├── notebooks
│   └── scripts
├── src
│   ├── AnalysisVSPAERO.py
│   ├── CST.py
│   ├── ISAspecification.py
│   ├── RollRudderGain.py
│   ├── TrimTurnSolver.py
│   ├── util.py
│   └── VvGammaChart.py
├── pyproject.toml
├── README.md
└── tree.ipynb


### ディレクトリだけ表示


In [5]:
tree(".", exclude=[".git", "node_modules", "__pycache__"], max_depth=3, show_files=False)

C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP
├── 99_old
│   ├── Pod Plane
│   ├── pyvsp.2026.06.28
│   │   ├── analysis
│   │   ├── geometry
│   │   └── util
│   ├── tests.2026.06.28
│   │   └── test_outputs
│   ├── tmp.2026.06.28
│   └── VSPU Tutorial Models
│       ├── Chapter 1 - VSP Fundamentals
│       ├── Chapter 2 - Modeling and Designing Intent
│       ├── Chapter 3 - Model Analysis
│       ├── Chapter 4 - Import and Export
│       ├── Chapter 5 - Advanced Methods
│       ├── Common Models
│       └── Section Backgrounds
├── docs
├── examples
│   ├── models
│   │   ├── DG1000s
│   │   ├── DG300
│   │   ├── G102
│   │   └── G103A
│   ├── notebooks
│   │   └── vv_gamma_cases
│   └── scripts
└── src


### ASCII 文字だけで表示

ターミナルやフォント環境によって罫線が崩れる場合に使います。


In [6]:
tree(".", exclude=[".git", "node_modules", "__pycache__"], max_depth=2, ascii_only=True)

C:\Users\mtkbirdman\OneDrive\Documents\OpenVSP
+-- 99_old
|   +-- Pod Plane
|   +-- pyvsp.2026.06.28
|   +-- tests.2026.06.28
|   +-- tmp.2026.06.28
|   +-- VSPU Tutorial Models
|   +-- AnalysisVSPAERO.2026.07.03.py
|   +-- TurnTrim.2026.07.03.py
|   `-- VVGammaChart.2026.07.03.py
+-- docs
+-- examples
|   +-- models
|   +-- notebooks
|   `-- scripts
+-- src
|   +-- AnalysisVSPAERO.py
|   +-- CST.py
|   +-- ISAspecification.py
|   +-- RollRudderGain.py
|   +-- TrimTurnSolver.py
|   +-- util.py
|   `-- VvGammaChart.py
+-- pyproject.toml
+-- README.md
`-- tree.ipynb


## 注意点

- `exclude` はフルパスではなく、名前一致です。
  - 例：`exclude=["node_modules"]` は、どの階層にある `node_modules` も除外します。
- `show_hidden=False` のとき、`.git` や `.env` のようなドット始まりの項目は表示しません。
- Linux `tree` コマンドの完全互換ではありません。
- コマンドラインから日常的に使う場合は、この関数を `.py` に移して CLI 化する方が自然です。
